In [1]:
import os
import time
import praw
import pandas as pd
from dotenv import load_dotenv
from urllib.parse import urlparse
load_dotenv()

NOW = time.time()
ONE_DAY = 86400

https://searchoperatorsguide.com/reddit-search-operators/

In [2]:
# Setup reddit scraper
CLIENT_ID = os.getenv("client_id")
CLIENT_SECRET = os.getenv("client_secret")

reddit = praw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    user_agent="script:italy_url_tracker:v1.0",
)

In [3]:
# cleanup urls and extract comments recursively to keep structure for now.
from urllib.parse import urlparse, urlunparse

def normalize_url(u):
    try:
        parsed = urlparse(u.lower())

        # remove query params + fragments
        clean = parsed._replace(
            query="",
            fragment=""
        )

        path = clean.path.rstrip("/")

        # optional: remove .html
        path = path.replace(".html", "")

        clean = clean._replace(path=path)

        return urlunparse(clean)

    except Exception:
        return u.lower()

def extract_comment(comment, depth=0, max_depth=10):
    """Recursively extract comment tree"""
    try:
        if depth > max_depth:
            return None

        return {
            "id": comment.id,
            "author": str(comment.author),
            "body": comment.body,
            "score": comment.score,
            "created_utc": comment.created_utc,
            "replies": [
                child for child in (
                    extract_comment(reply, depth + 1, max_depth)
                    for reply in comment.replies
                ) if child is not None
            ],
        }
    except Exception as e:
        return {
            "id": getattr(comment, "id", None),
            "error": str(e)
        }

In [4]:
import json
file_path = "data/202604_21-28/output_20260421.jsonl"
# Regex pattern for 20260421 into 2026-04-21
date = "2026-04-21"
with open(file_path, "r") as f:
    data = [json.loads(line) for line in f]
df = pd.DataFrame(data)
# Remove zazzom
df = df[df["MentionSourceName"] != "zazoom.it"].reset_index(drop=True)
df['date'] = [date] * len(df)
df['date'] = pd.to_datetime(df['date'])

In [5]:
# Clean urls (remove query params, fragments, .html, etc.) and extract keywords for all unique urls
df['clean_urls'] = df['MentionIdentifier'].apply(normalize_url)

In [12]:
df_forsearch = df[["clean_urls"]].drop_duplicates().reset_index(drop=True)

In [ ]:
import time
from prawcore.exceptions import TooManyRequests


def search_subreddits(
    subreddits_str,
    df,
    reddit,
    time_filter="month",
    limit=50,
    max_retries=5,
    base_sleep=30):

    results = []

    for idx, row in df.iterrows():

        url = row["clean_urls"]
        query = f"url:{url}"
        success = False

        for attempt in range(max_retries):
            try:
                submissions = reddit.subreddit(subreddits_str).search(
                    query,
                    sort="new",
                    time_filter=time_filter,
                    limit=limit
                )
                submissions = list(submissions)
                for submission in submissions:
                    print(
                        f"\tSubmission subreddit: {submission.subreddit}, "
                        f"title: {submission.title}, "
                        f"permalink: {submission.permalink}"
                    )
                    results.append(submission)
                success = True
                # added a small pause between normal requests to try and avoid the rate limit
                time.sleep(2)
                break 

            except TooManyRequests as e:
                # Use Reddit retry_after if available
                retry_after = getattr(e, "retry_after", None)
                if retry_after is not None:
                    wait_time = retry_after
                else:
                    wait_time = base_sleep * (2 ** attempt)
                print(
                    f"429 TooManyRequests for {url} "
                    f"(attempt {attempt + 1}/{max_retries}) "
                    f"-> sleeping {wait_time}s"
                )
                time.sleep(wait_time)

            except Exception as e:
                print(f"Error for {url}: {e}")
                time.sleep(5)
                break

        if not success:
            print(f"FAILED after {max_retries} retries: {url}")

    return results

In [ ]:
def extract_submission_data(results, max_depth=10):
    final_results = []
    for submission in results:
        output_json = {}
        # General info
        output_json["id"] = submission.id
        output_json["subreddit"] = str(submission.subreddit)
        output_json["permalink"] = submission.permalink
        output_json["created_utc"] = submission.created_utc

        output_json["author"] = str(submission.author)


        output_json["title"] = submission.title
        output_json["selftext"] = submission.selftext
        output_json["url"] = submission.url

        output_json["comment"] = [extract_comment(comment, max_depth) for comment in submission.comments]
        final_results.append(output_json)
    return final_results
        

In [ ]:
subreddits_str = "italy+italia"
results_italyitalia = search_subreddits(subreddits_str,
                                        df,
                                        reddit,
                                        time_filter="month",
                                        limit=50,
                                        max_retries=5,
                                        base_sleep=30)

In [ ]:
final_results_italyitalia = extract_submission_data(results_italyitalia, max_depth=10)
output_path = "data/test/2026-04-21_italyitalia.jsonl"
with open(output_path, "w") as f:
    for item in final_results_italyitalia:
        f.write(json.dumps(item) + "\n")

In [ ]:
# Hand crated subreddits, info from:
# https://www.reddit.com/r/Italia/comments/o6nbkf/elenco_dei_subreddit_italiani/?tl=it
# https://github.com/danieleongari/awesome-italian-reddit (first 200)

In [ ]:
subreddits_generali = [
    "italy", "italia", "ItaliaRossa", "politicaITA", "oknotizie", "EconomiaItaliana", "innovazione" 
]

subreddits_regionali = [
    "Abruzzo", "Basilicata", "Calabria", "Emilia_Romagna", "Friuli", "Liguria",
    "Lombardia", "Marche", "Molise", "Piemonte", "Puglia", "Sardegna", "Sicilia", 
    "Toscana", "Trentino_alto_Adige", "Umbria", "Veneto"
]

subreddits_locali = [
    "Bologna", "Firenze", "Milano", "Napoli", "Padova", "Roma", "Torino", "Modena", 
    "Trieste", "Genova", "Bari", "Catania", "Siracusa", "Trento", "Perugia", "Aosta", 
    "Venezia", "brescia", "cagliari", "parma"
]

In [ ]:
subreddits_generali_str = "+".join(subreddits_generali)
subreddits_regionali_str = "+".join(subreddits_regionali)
subreddits_locali_str = "+".join(subreddits_locali)

In [ ]:
results_generali = search_subreddits(subreddits_generali_str,
                                        df,
                                        reddit,
                                        time_filter="month",
                                        limit=50,
                                        max_retries=5,
                                        base_sleep=30)

final_results_generali = extract_submission_data(results_generali, max_depth=10)
output_path = "data/test/2026-04-21_generali.jsonl"
with open(output_path, "w") as f:
    for item in final_results_generali:
        f.write(json.dumps(item) + "\n")

In [ ]:
results_regionali = search_subreddits(subreddits_regionali_str,
                                        df,
                                        reddit,
                                        time_filter="month",
                                        limit=50,
                                        max_retries=5,
                                        base_sleep=30)

final_results_regionali = extract_submission_data(results_regionali, max_depth=10)
output_path = "data/test/2026-04-21_regionali.jsonl"
with open(output_path, "w") as f:
    for item in final_results_regionali:
        f.write(json.dumps(item) + "\n")

In [ ]:
results_locali = search_subreddits(subreddits_locali_str,
                                        df,
                                        reddit,
                                        time_filter="month",
                                        limit=50,
                                        max_retries=5,
                                        base_sleep=30)

final_results_locali = extract_submission_data(results_locali, max_depth=10)
output_path = "data/test/2026-04-21_locali.jsonl"
with open(output_path, "w") as f:
    for item in final_results_regionali:
        f.write(json.dumps(item) + "\n")

- - - 
OLD FUNCTIONS

In [ ]:
final_results

In [ ]:
# Search with clean url directly (subreddits regionali)
results = []
for idx, row in df_forsearch.iterrows():
    url = row['clean_urls']
    query = f"url:{url}"
    submissions = reddit.subreddit(subreddits_regionali_str).search(query, 
                                                          sort="new", 
                                                          time_filter="month", 
                                                          limit=100)
    for submission in submissions:
        print(f"\tSubmission subreddit: {submission.subreddit}, title: {submission.title}, permalink: {submission.permalink}")
        results.append(submission)
print(f"\tTotal found: {len(results)}")

final_results = []
for submission in results:
    output_json = {}
    # General info
    output_json["id"] = submission.id
    output_json["subreddit"] = str(submission.subreddit)
    output_json["permalink"] = submission.permalink
    output_json["created_utc"] = submission.created_utc
    
    # Submission info
    output_json["title"] = submission.title
    output_json["selftext"] = submission.selftext
    output_json["url"] = submission.url

    # Comments
    output_json["comment"] = [extract_comment(comment, max_depth=10) for comment in submission.comments]

    final_results.append(output_json)

In [ ]:
final_results

In [ ]:
# Search with clean url directly (subreddits locali)
results = []
for idx, row in df_forsearch.iterrows():
    url = row['clean_urls']
    query = f"url:{url}"
    submissions = reddit.subreddit(subreddits_locali_str).search(query, 
                                                          sort="new", 
                                                          time_filter="month", 
                                                          limit=100)
    for submission in submissions:
        print(f"\tSubmission subreddit: {submission.subreddit}, title: {submission.title}, permalink: {submission.permalink}")
        results.append(submission)
print(f"\tTotal found: {len(results)}")

final_results = []
for submission in results:
    output_json = {}
    # General info
    output_json["id"] = submission.id
    output_json["subreddit"] = str(submission.subreddit)
    output_json["permalink"] = submission.permalink
    output_json["created_utc"] = submission.created_utc
    
    # Submission info
    output_json["title"] = submission.title
    output_json["selftext"] = submission.selftext
    output_json["url"] = submission.url

    # Comments
    output_json["comment"] = [extract_comment(comment, max_depth=10) for comment in submission.comments]

    final_results.append(output_json)

In [ ]:
final_results

#### Alcune osservazioni
- Ci sono articoli "vecchi"
- Ci sono articoli che non compaiono nel dump di gdelt

In [7]:
df_ilpost = df[df["MentionSourceName"] == "ilpost.it"].reset_index(drop=True)
# Drop duplicate rows with same clean url
df_ilpost = df_ilpost.drop_duplicates(subset=["clean_urls"]).reset_index(drop=True)

In [142]:
# Search with clean url directly
found = 0
for idx, row in df_ilpost.iterrows():
    url = row['clean_urls']
    print(f"Searching for: {url}")
    query = f"url:{url}"
    submissions = reddit.subreddit("italy+italia").search(query, 
                                                          sort="new", 
                                                          time_filter="month", 
                                                          limit=100)
    for submission in submissions:
        print(f"\tSubmission subreddit: {submission.subreddit}, title: {submission.title}, permalink: {submission.permalink}")
        found += 1
print(f"\tTotal found: {found}")

Searching for: https://www.ilpost.it/2023/01/30/come-si-vive-al-41-bis
Searching for: https://www.ilpost.it/2026/04/21/le-prime-pagine-di-oggi-4540
Searching for: https://www.ilpost.it/2026/04/20/propaganda-sarcasmo-regime-iran-guerra
Searching for: https://www.ilpost.it/2026/04/15/papa-leone-xiv-donald-trump-scontri-stati-uniti
Searching for: https://www.ilpost.it/2026/02/20/domande-risposte-sentenza-dazi-illegittimi-corte-suprema-trump
Searching for: https://www.ilpost.it/2026/02/20/corte-suprema-sentenza-dazi-trump
Searching for: https://www.ilpost.it/2026/04/19/louisiana-shreveport-uomo-ucciso-otto-bambini
Searching for: https://www.ilpost.it/2024/02/08/lavoro-carcere-recidiva
Searching for: https://www.ilpost.it/2026/04/19/cina-iran-sostegno-militare
Searching for: https://www.ilpost.it/2026/04/17/guida-cessate-il-fuoco-iran-stati-uniti-israele-libano
Searching for: https://www.ilpost.it/2026/04/20/trump-minacce-crimini-di-guerra-iran
Searching for: https://www.ilpost.it/2026/04/2

Ci sono articoli piu vecchi nel dump di gdelt, tipo questo è vecchio di 3 mesi (vedi febbraio)

In [169]:
import datetime
submissions = reddit.subreddit("italy+italia").search("url:ilpost", 
                                                          sort="new", 
                                                          time_filter="month", 
                                                          limit=100)
cnt = 0
for submission in submissions:
    print(f"\tSubmission subreddit: {submission.subreddit}, "
          f"title: {submission.title}, "
          f"permalink: {submission.permalink}, "
          f"date: {datetime.datetime.utcfromtimestamp(submission.created_utc)}")
    cnt += 1
print(f"Total submissions found: {cnt}")

	Submission subreddit: Italia, title: Il ministro Giuli ha dato dell'assenteista al ministro Salvini, permalink: /r/Italia/comments/1t72t7m/il_ministro_giuli_ha_dato_dellassenteista_al/, date: 2026-05-08 09:25:32
	Submission subreddit: italy, title: Il video in cui Valditara dice che Piersanti Mattarella è stato ucciso dalle Brigate Rosse, permalink: /r/italy/comments/1t71or6/il_video_in_cui_valditara_dice_che_piersanti/, date: 2026-05-08 08:20:40
	Submission subreddit: Italia, title: Il video in cui Valditara dice che Piersanti Mattarella è stato ucciso dalle Brigate Rosse, permalink: /r/Italia/comments/1t71dvb/il_video_in_cui_valditara_dice_che_piersanti/, date: 2026-05-08 08:03:02
	Submission subreddit: italy, title: L'India è sempre più a forma di Narendra Modi, permalink: /r/italy/comments/1t5dzzc/lindia_è_sempre_più_a_forma_di_narendra_modi/, date: 2026-05-06 14:07:33
	Submission subreddit: Italia, title: In ricordo di Osso (di Michele Serra), permalink: /r/Italia/comments/1t4flp

/tmp/ipykernel_109102/813521348.py:11: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  f"date: {datetime.datetime.utcfromtimestamp(submission.created_utc)}")


Some are missing from GDELT dump it is not exhaustive

Submission subreddit: italy, title: Il sottosuolo di Napoli sembra fatto apposta per le bande criminali, permalink: /r/italy/comments/1ss17tu/il_sottosuolo_di_napoli_sembra_fatto_apposta_per/, date: 2026-04-21 21:21:05


Submission subreddit: italy, title: Sempre meglio essere gentili con i chatbot, permalink: /r/italy/comments/1srr7c0/sempre_meglio_essere_gentili_con_i_chatbot/, date: 2026-04-21 15:33:00
	

### Structure of submission object

In [129]:
print(submission.subreddit)
print(submission.title)
print(submission.name)
print(submission.category)
print(submission.selftext_html)
print(submission.permalink)
print(submission.url)

# All trees of comments
print(extract_comment(submission.comments[0], max_depth=2))
print(extract_comment(submission.comments[1], max_depth=2))

italy
La Corte Suprema degli Stati Uniti ha giudicato illegali i dazi di Trump
t3_1r9y851
None
None
/r/italy/comments/1r9y851/la_corte_suprema_degli_stati_uniti_ha_giudicato/
https://www.ilpost.it/2026/02/20/corte-suprema-sentenza-dazi-trump/?utm_medium=social&utm_source=twitter&utm_campaign=lancio
{'id': 'o6fpzns', 'author': 'Klutzy-Weakness-937', 'body': 'Quali saranno le conseguenze?', 'score': 159, 'created_utc': 1771601260.0, 'replies': [{'id': 'o6fqceb', 'author': 'Radagast92', 'body': ">La sentenza ha tre conseguenze concrete di enorme portata, anche se non si sa ancora\xa0*come*\xa0queste verranno messe in pratica.\xa0La prima è il fatto che con ogni probabilità il governo dovrà restituire alle imprese che hanno pagato i dazi tutto quello che ha incassato. Secondo gli\xa0[ultimi dati disponibili](https://fred.stlouisfed.org/series/B235RC1Q027SBEA), si tratta di centinaia di miliardi di dollari.\n\n>La seconda conseguenza è che cambieranno di nuovo e in modo sostanziale le regol